In [1]:
print("ok")

ok


# Trace The Ace V2

## Stage 01 — Master Dataset

Goal:
Create a leakage-safe, response-level master dataset from the official competition files.

Input:
- train_transcripts/
- train_features.csv
- train_labels.csv

Output:
- master_train.parquet
- master_test.parquet

Notes:
- One row = one response
- Preserve all raw information
- No feature engineering
- No preprocessing beyond data integration

In [2]:
# ==========================================================
# Trace The Ace V2
# Stage 01 - Master Dataset Creation
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import json
import os

# ==========================================================
# CONFIG
# ==========================================================

DATASET = Path("/kaggle/input/datasets/shri7ul/trace-the-race-competition-dataset/Trace-The-Race-Dataset")

TRAIN_FEATURES = DATASET / "train_features_TMQTWsB.csv"
TRAIN_LABELS = DATASET / "train_labels_44ujmj2.csv"
TRAIN_TRANSCRIPTS = DATASET / "train_transcripts"

OUTPUT_DIR = Path("/kaggle/working/master")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATASET)

/kaggle/input/datasets/shri7ul/trace-the-race-competition-dataset/Trace-The-Race-Dataset


## Load Official Files

In [3]:
# ==========================================================
# Load official datasets
# ==========================================================

train_features = pd.read_csv(TRAIN_FEATURES)
train_labels = pd.read_csv(TRAIN_LABELS)

print(f"Train Features : {train_features.shape}")
print(f"Train Labels   : {train_labels.shape}")

Train Features : (35072, 4)
Train Labels   : (35072, 2)


In [4]:
# ==========================================================
# Build transcript from one session
# ==========================================================

def build_session(file_path):

    df = pd.read_csv(file_path)

    df = df.sort_values("utterance_id")

    transcript = []
    student = []
    tutor = []
    background = []

    for _, row in df.iterrows():

        role = str(row["role"]).strip().upper()
        text = str(row["content"]).strip()

        transcript.append(f"[{role}] {text}")

        if role == "STUDENT":
            student.append(text)

        elif role == "TUTOR":
            tutor.append(text)

        else:
            background.append(text)

    return {

        "session_id": df["session_id"].iloc[0],

        "transcript": "\n".join(transcript),

        "student_text": " ".join(student),

        "tutor_text": " ".join(tutor),

        "background_text": " ".join(background)

    }

In [5]:
# ==========================================================
# Read all transcript files
# ==========================================================

sessions = []

files = sorted(TRAIN_TRANSCRIPTS.glob("*.csv"))

for file in tqdm(files):

    sessions.append(build_session(file))

session_df = pd.DataFrame(sessions)

print(session_df.shape)

session_df.head()

  0%|          | 0/22821 [00:00<?, ?it/s]

(22821, 5)


,session_id,transcript,student_text,tutor_text,background_text
0,aaaedit,[TUTOR] Hello?\n[BACKGROUND] [unclear]\n[BACKG...,"Good. Good. During the weekend, on the 15th of...","Hello? Hi, Lachlan. How are you doing today? O...","[unclear] Hello. Yeah, we can continue. We can..."
1,aaaptjd,[BACKGROUND] [unclear]\n[TUTOR] Hello?\n[STUDE...,Hello? I'm good. Adding and subtracting amount...,"Hello? Hi, how are you doing today? Okay, I re...",[unclear] Is it possible for us to have 95p le...
2,aabkeov,[STUDENT] Hello.\n[BACKGROUND] [unclear]\n[TUT...,"Hello. Hi. Thank you. Hello? Yeah, how are you...","Hello. Welcome back. Hi, Kaelan. [unclear] Hi....",[unclear] Yeah. Ratio means— Does she have 10?...
3,aacggvb,"[BACKGROUND] [unclear]\n[TUTOR] Hello? Tobias,...","Uh, yeah. Yes. Good. Huh? Yeah, that's why it'...","Hello? Tobias, can you hear me? Okay. So we ha...","[unclear] First of all, so when you're writing..."
4,aadexbc,"[BACKGROUND] [unclear]\n[TUTOR] Hi. Bryony, ca...","Hello. I'm good, how are you? Good. Yeah. Yes....","Hi. Bryony, can you hear me? Hi Bryony, how ar...","[unclear] Okay. So— [unclear] Okay, yes. 40, 4..."


In [6]:
# ==========================================================
# Merge Features
# ==========================================================

master_train = train_features.merge(
    session_df,
    on="session_id",
    how="left"
)

master_train = master_train.merge(
    train_labels,
    on="response_id",
    how="left"
)

print(master_train.shape)

(35072, 9)


In [7]:
# ==========================================================
# Validation
# ==========================================================

print("="*50)

print("Missing Transcript")

print(master_train["transcript"].isna().sum())

print()

print("Duplicate response_id")

print(master_train["response_id"].duplicated().sum())

print()

print("Duplicate session_id")

print(master_train["session_id"].duplicated().sum())

print("="*50)

Missing Transcript
0

Duplicate response_id
0

Duplicate session_id
12251


In [8]:
# ==========================================================
# Save parquet
# ==========================================================

master_train.to_parquet(
    OUTPUT_DIR / "master_train.parquet",
    index=False
)

metadata = {

    "rows": int(master_train.shape[0]),

    "columns": list(master_train.columns)

}

with open(OUTPUT_DIR / "dataset_info.json","w") as f:

    json.dump(metadata,f,indent=4)

print("Saved Successfully.")

Saved Successfully.


In [9]:
metadata = {

    "rows": int(master_train.shape[0]),

    "columns": list(master_train.columns),

    "num_sessions": int(master_train["session_id"].nunique()),

    "num_responses": int(master_train["response_id"].nunique()),

    "created_by": "Trace The Ace V2",

    "version": "1.0"

}